In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_community.document_loaders import TextLoader
loader=TextLoader("speech.txt")

docs1=loader.load()

In [ ]:
#Text Splitting Strategies

from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

token_split=TokenTextSplitter(chunk_size=50,chunk_overlap=10)

In [ ]:
token_text=token_split.split_documents(docs1)

In [ ]:
from langchain_openai import OpenAIEmbeddings
#Documents to embeddings 
embeddings_1024=OpenAIEmbeddings(model="text-embedding-3-large",dimensions=1024)
#It will directly convert to embeddings
#Store Embeddings to VectorDB(chromadb)

persist_directory="./chroma_db"
from langchain_community.vectorstores import Chroma
vectorstore=Chroma.from_documents(token_text,embeddings_1024,persist_directory=persist_directory)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to :{persist_directory}")

Vector store created with 38 vectors
Persisted to :./chroma_db


In [ ]:
#Test Similarity Search

query="conduct ourselves as belligerents in a high spirit of right and fairness"
similar_docs=vectorstore.similarity_search(query,k=1)

similar_docs

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Document(page_content=' fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire', metadata={'source': 'speech.txt'})]

In [ ]:
#Advanced Similarity Search
results_score=vectorstore.similarity_search_with_score(query,k=3)
results_score

[(Document(page_content=' fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire', metadata={'source': 'speech.txt'}),
  0.5179236761634753),
 (Document(page_content=' fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire', metadata={'source': 'speech.txt'}),
  0.5179236782820574),
 (Document(page_content=' all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…\n\nIt will', metadata={'source': 'speech.txt'}),
  0.5793344306939431)]

In [ ]:
#Initalizing the LLM

from langchain_openai import ChatOpenAI
llm=ChatOpenAI(
    model_name="gpt-3.5-turbo"
)

In [ ]:
test_response=llm.invoke("What is Large language model")
test_response

AIMessage(content="A large language model is a type of artificial intelligence model that has been trained on vast amounts of text data in order to understand and generate human language. These models are able to process and generate text in a way that is increasingly indistinguishable from human-written text. Large language models are used in a wide range of applications, including natural language processing, text generation, and language translation. Examples of large language models include OpenAI's GPT-3 and Google's BERT.", response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 12, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-249a9315-34f1-401b-bbc4-b88ebe64775f-0')

In [ ]:
from langchain.chains.retrieval import create_retrieval_chain

from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
#Convert vectorstore to retriver
#retriever is an interface which interacts with vector store

retriever=vectorstore.as_retriever(
    search_kwargs={"k":3}#Retrieve top 3 documents
)

retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000211315C32E0>, search_kwargs={'k': 3})

In [ ]:
#Create Prompt Template

#What ever retrived context from vectorstore it is placed in context

from langchain_core.prompts import ChatPromptTemplate


system_prompt="""You are assistant for question answering tasks
  if you don't the answer just say don't know


  Context"{context}
"""

prompt=ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}")
    ]

    
)


In [ ]:
#Whatever relvantinformation we are getting from vectorstore it combine by using document chain with model 
#pushes to context

#This chain works as
#takes retreived documents
#Stuff's them into the prompt's {context} placeholder
#Sends the complete prompt to the LLM
#Returns the LLM's response

from langchain.chains.combine_documents import create_stuff_documents_chain

document_chain=create_stuff_documents_chain(llm,prompt)

document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), config={'run_name': 'format_inputs'})
| ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template='You are assistant for question answering tasks\n  if you don\'t the answer just say don\'t know\n\n\n  Context"{context}\n')), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x00000211432ABA90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000211432AB6D0>, openai_api_key=SecretStr('**********'), openai_proxy='')
| StrOutputParser(), config={'run_name': 'stuff_documents_chain'})

In [ ]:
from langchain.chains.retrieval import create_retrieval_chain

rag_chain=create_retrieval_chain(retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000211315C32E0>, search_kwargs={'k': 3}), config={'run_name': 'retrieve_documents'})
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), config={'run_name': 'format_inputs'})
            | ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template='You are assistant for question answering tasks\n  if you don\'t the answer just say don\'t know\n\n\n  Context"{context}\n')), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])
            | ChatOpenAI(client=<openai.reso

In [ ]:
response=rag_chain.invoke({"input":"conduct ourselves as belligerents in a high spirit of right and fairness"})

In [ ]:
response['answer']

'conduct ourselves as belligerents in a high spirit of right and fairness'

In [ ]:
#Create flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableParallel

#create custom prompt template
#Both from_template and from_messages will work same
#If you want more systematic and role based use from_messages
prompt=ChatPromptTemplate.from_template(

    
    """ Use the following context and answer the follwing questions
    If you don't know the answer say no and provide 
    specific details to support the answer
    Context: {context}
    Question:{question}
    Answer:"""
)

In [ ]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000211315C32E0>, search_kwargs={'k': 3})

In [ ]:
#Format the outpuit documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for  doc in docs)


In [ ]:
#Build the LCEL chain
#Runnable Pass Through passes input as it is
rag_chain_lcel=(
    {"context":retriever| format_docs,
    "question":RunnablePassthrough()
    }

    |prompt
    |llm
    |StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000211315C32E0>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template=" Use the following context and answer the follwing questions\n    If you don't know the answer say no and provide \n    specific details to support the answer\n    Context: {context}\n    Question:{question}\n    Answer:"))])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x00000211432ABA90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000211432AB6D0>, openai_api_key=SecretStr('**********'), openai_proxy='')
| StrOutputParser()

In [ ]:
rag_chain_lcel.invoke("conduct ourselves as belligerents in a high spirit of right and fairness")

'Yes, the speaker expresses confidence in conducting themselves as belligerents in a high spirit of right and fairness.'

In [ ]:
retriever.get_relevant_documents("conduct ourselves as belligerents in a high spirit of right and fairness")

[Document(page_content=' fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire', metadata={'source': 'speech.txt'}),
 Document(page_content=' fighting for.\n\n…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire', metadata={'source': 'speech.txt'}),
 Document(page_content=' all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…\n\nIt will', metadata={'source': 'speech.txt'})]

In [ ]:
#Adding new document to vector db

from langchain.schema import Document
new_doc=Document(
    page_content="new_document",
    metadata={"source":"manual_addition","topic":"reinforcement_learning"}
)

In [ ]:
new_doc

Document(page_content='new_document', metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'})

In [ ]:


from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=10,chunk_overlap=2)

new_chunks=text_splitter.split_documents([new_doc])

In [ ]:
new_chunks

[Document(page_content='new_docume', metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'}),
 Document(page_content='ment', metadata={'source': 'manual_addition', 'topic': 'reinforcement_learning'})]

In [ ]:
#Adding documents to chromadb

vectorstore.add_documents(new_chunks)

['2e9c54a7-3fbb-47c5-b5c3-992ef1d65654',
 '0a2dcdd4-b816-478b-96c2-6531e0ba02c9']

In [ ]:
vectorstore._collection.count()

40

In [ ]:
#Adding Conversational Memory

#create_history_aware_retriever :- Makes the retriever understand conversation context
#Message Placeholder:Placeholder for chat history in prompts
#Human Message/AI Message:Structured message types for conversation history


from langchain.chains import create_history_aware_retriever
from langchain_core import chat_history
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import AIMessage,HumanMessage

contextulize_q_system_prompt="""GIven a chat history and the latest user question 
which might reference context in the chat history,formulate a standalone question
which can be understood without the chat history.Do not answer the question,just reformulate
it if needed and otherwise return as it is"""


contextulize_q_prompt=ChatPromptTemplate.from_messages(
    [
        ("system",contextulize_q_system_prompt),
         MessagesPlaceholder("chat_history"),
        ("human","{input}")
    ]
)


In [ ]:
#create history aware
#It rewrites the question based on previous context

history_aware_retriever=create_history_aware_retriever(
    llm,retriever,contextulize_q_prompt
)

history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000211315C32E0>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]]}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='GIven a chat history and the latest user question \nwhich might reference context in the chat history,formulate a standalone question\nwhich can be understood without the chat history.Do not answer 

In [ ]:
system_prompt= """Your are an AI assitant answer the question 
using relevant chunks if you dont know the answer say no.

Context:{context}"""

prompt=ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human","{input}")
    ]
)


In [ ]:
question_answer_chain=create_stuff_documents_chain(llm,prompt)

#create conversational RAG chain
#For the first time there is no previous context so it acts as retriver\
#It gives relevant documents and it will stuff to prompt 

#For second time history_retriver will rewrite question and get 
# relevant documents and stuff to prompt
conversation_rag_chain=create_retrieval_chain(
    history_aware_retriever,question_answer_chain
)

In [ ]:
conversation_rag_chain.invoke(
    {
          "chat_history":chat_history,
           "input":"What is LLM"
    }
    
)

In [ ]:
from pinecone import pinecone

In [1]:
#MMR
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI

from langchain_core.prompts import PromptTemplate
from langchain.chains.retrieval import create_retrieval_chain
#from langchain.chains.combine_documents import create_stuff_documents_chain

In [2]:
from dotenv import  load_dotenv
from langchain_openai import ChatOpenAI
import os
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")


os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
llm=ChatOpenAI(model="gpt-4o")

In [3]:
#load text file
loader=TextLoader("text.txt")

raw_docs=loader.load()
splits=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

doc_splits=splits.split_documents(raw_docs)

In [10]:
#user query

input="InfoWorld described LangChain as a software development kit"

from langchain_community.vectorstores import FAISS

from langchain_core import retrievers
from langchain_openai import OpenAIEmbeddings
embedding_model=OpenAIEmbeddings()

vector_Store=FAISS.from_documents(doc_splits,embedding_model)
retriever=vector_Store.as_retriever(search_type="mmr",search_kwargs={"k":8})
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E5486D3640>, search_type='mmr', search_kwargs={'k': 8})

In [11]:
prompt=PromptTemplate.from_template(
    """Answer the question based on the context below

    input:"{input}"

    context:{context}

   

    """
)

In [12]:
llm=ChatOpenAI(model="gpt-4o")

In [15]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
document_chain=prompt|llm|parser
chain=create_retrieval_chain(retriever,document_chain)

retriever_docs=retriever.invoke(user_query)
retriever_docs

doc_lines=[f"{index+1}. {doc.page_content}" for index, doc in enumerate(retriever_docs)]
formated_docs="\n".join(doc_lines)

formated_docs

'1. synthetic data generation.[14] InfoWorld described LangChain as a software development kit that simplifies the connection between large language models and external applications through a unified\n2. April 2023, it can read from more than 50 document types and data sources.[21]\n3. Langchain is a framework for building applications with LLM.\nLangchain provides modular abstraction to combine llms\nYou can create chains,agents,memory and retriever\n4. and pymupdf for PDF file text extraction and manipulation; Python and JavaScript code generation, analysis, and debugging; Milvus vector database[18] to store and retrieve vector embeddings; Weaviate\n5. As of March 2023, LangChain included integrations with systems including Amazon, Google, and Microsoft Azure cloud storage;[16] API wrappers for news, movie information, and weather; Bash for\n6. to change the underlying model without major code changes.[15]\n7. including for logging; text mapping for k-nearest neighbors search;\n8. fe

In [16]:
response=chain.invoke(
    {
        "input":input,
        "context": formated_docs
    }
)

In [21]:
response['answer']

'InfoWorld described LangChain as a software development kit that simplifies the connection between large language models and external applications through a unified framework.'

In [22]:
#Query Expansion

query_expansion=PromptTemplate.from_template("""
You are a helpful assistant .Expand the following query to improve document retrieval by adding  relevant documents
                                             
                                            
input:{input}
Expanded query:
""" )

In [23]:
query_chain=query_expansion|llm|StrOutputParser()

In [24]:
query_expansion_chain=query_chain.invoke({"input":"Langchain memory"})

In [25]:
query_expansion_chain

'To expand the query "Langchain memory" for improved document retrieval, we\'ll add relevant terms and concepts related to "Langchain" and "memory" that can help capture more comprehensive and varied documents. Here\'s the expanded query:\n\n"Langchain memory" AND ("Langchain framework" OR "Langchain technology" OR "cognitive memory models" OR "neural networks" OR "AI memory systems" OR "memory optimization" OR "cognitive computing" OR "memory algorithms" OR "Langchain applications" OR "AI learning systems" OR "memory management" OR "Langchain use cases" OR "machine learning" OR "intelligent systems" OR "memory enhancement")\n\nThis expanded query includes synonyms, related technologies, potential applications, and contexts in which "Langchain memory" might be relevant. This approach should improve the retrieval of documents related to the initial query by broadening the search scope.'